# quality

> which documents in a store are retrieval noise, scored from the vectors already in the file

In [ ]:
#| default_exp quality

No labels, no model, no second pass over the corpus. The features are document-level
aggregates of quantities the chunk store already holds, and the blend of them scores **0.988 AUC**
against hand-marked junk.

The vector legs answer "what is this like"; these answer "is this worth retrieving at all". A
cookie banner, a nav sidebar and a licence header are near-duplicates of every other page's, sit
far from the corpus mean, and are made of words that are everywhere.

In [ ]:
#| export
import json, math, re, time, warnings
from collections import Counter, defaultdict
import numpy as np
from fastcore.all import AttrDict, L, patch
from fastlite import Database
from litesearch.core import _np_dtype

## The vectors, as one array

In [ ]:
#| export
#: Features oriented larger=noisier; document-level aggregates of chunk quantities.
NOISE_FEATURES = ('hub', 'dup_out', 'off_centre', 'spread_chunk', 'spread_doc', 'low_idf',
                  'redundancy', 'short', 'promiscuity')

#: Default blend from per-feature AUCs (`evals/noise.py`); `spread_doc` is 0.54 so weight 0.
NOISE_W = dict(hub=1.0, spread_chunk=0.8, dup_out=0.8, off_centre=0.5, low_idf=0.3,
               promiscuity=0.5, redundancy=0.1, spread_doc=0.0, short=0.0)

@patch
def chunk_matrix(self:Database,
                 store:str='store',   # the chunk table
                 limit:int=None,      # sample this many chunks instead of using all of them
                 seed:int=0,
                 dtype=np.float16,    # the width the store holds
                 ) -> tuple:
    """`(ids, doc_ids, texts, V)`: every chunk's stored vector, L2-normalised, as one array."""
    rows = [r for r in self.t[store](select='id, doc_id, content, embedding') if r['embedding']]
    if limit and len(rows) > limit:
        idx = np.random.default_rng(seed).choice(len(rows), limit, replace=False)
        rows = [rows[i] for i in sorted(idx)]
    if not rows: return L(), L(), L(), np.zeros((0, 1), np.float32)
    V = np.frombuffer(b''.join(r['embedding'] for r in rows), dtype=dtype).reshape(len(rows), -1).astype(np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-9
    return (L(r['id'] for r in rows), L(r['doc_id'] for r in rows),
            L(r['content'] or '' for r in rows), V)

## Nearest neighbours, centroids and the rest of the arithmetic

In [ ]:
#| export
def _knn(V:np.ndarray, k:int=10, exact_max:int=30_000, block:int=1024) -> tuple:
    "`(idx, sim)` of each row's `k` nearest others. Exact by blocked matmul, HNSW past `exact_max`."
    n = len(V)
    k = min(k, n - 1)
    if n < 3 or k < 1: return np.zeros((n, 0), np.int32), np.zeros((n, 0), np.float32)
    if n > exact_max:
        from usearch.index import Index
        ix = Index(ndim=V.shape[1], metric='cos', dtype='f32')
        ix.add(np.arange(n), V)
        m = ix.search(V, count=k+1)
        return m.keys[:, 1:].astype(np.int32), (1.0 - m.distances[:, 1:]).astype(np.float32)
    idx, sim = np.empty((n, k), np.int32), np.empty((n, k), np.float32)
    for s in range(0, n, block):
        S = V[s:s+block] @ V.T
        for r in range(S.shape[0]): S[r, s+r] = -2.0        # never your own neighbour
        part = np.argpartition(-S, k, axis=1)[:, :k]
        rows = np.arange(S.shape[0])[:, None]
        ordr = np.argsort(-S[rows, part], axis=1)
        idx[s:s+block] = part[rows, ordr]
        sim[s:s+block] = S[rows, part[rows, ordr]]
    return idx, sim

def _centroids(V:np.ndarray, k:int=64, seed:int=0, iters:int=25) -> np.ndarray:
    """Topic centroids: seeded k-means++ then Lloyd on the unit sphere, L2-normalised."""
    k = max(2, min(k, len(V) // 4))
    if len(V) <= 8: return V.mean(0, keepdims=True) / (np.linalg.norm(V.mean(0)) + 1e-9)
    rng = np.random.default_rng(seed)
    C = np.empty((k, V.shape[1]), np.float32)
    C[0] = V[rng.integers(len(V))]
    d2 = 1.0 - V @ C[0]                                  # cosine distance, vectors are unit length
    for i in range(1, k):                                # k-means++: far points more likely to be picked
        pr = np.clip(d2, 0, None)**2
        tot = pr.sum()
        C[i] = V[rng.integers(len(V)) if tot <= 0 else int(np.searchsorted(np.cumsum(pr/tot), rng.random()))]
        d2 = np.minimum(d2, 1.0 - V @ C[i])
    prev = None
    for _ in range(iters):
        a = (V @ C.T).argmax(1)
        if prev is not None and np.array_equal(a, prev): break
        prev = a
        for i in range(k):
            m = a == i
            C[i] = V[m].sum(0) if m.any() else V[int(d2.argmax())]
        n = np.linalg.norm(C, axis=1, keepdims=True)
        C = C / np.where(n < 1e-9, 1.0, n)
        d2 = 1.0 - (V @ C.T).max(1)
    n = np.linalg.norm(C, axis=1)
    C = C[n > 1e-6]
    return C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)

_TOK = re.compile(r"[A-Za-z0-9_']+")
def chunk_idf(texts) -> dict:
    "Inverse document frequency over chunks: boilerplate is made of words that are everywhere."
    df, n = Counter(), max(len(texts), 1)
    for t in texts: df.update(set(_TOK.findall((t or '').lower())))
    return {w: math.log(n / (1 + c)) for w, c in df.items()}

def _entropy(P:np.ndarray) -> np.ndarray:
    "Normalised Shannon entropy of each row, in [0,1]; 1 means committing to nothing."
    P = np.clip(P, 1e-9, 1.0)
    return (-(P * np.log(P)).sum(1) / math.log(P.shape[1])) if P.shape[1] > 1 else np.zeros(len(P))

## The features

Nine of them, all oriented larger-is-noisier. `promiscuity` is the only one that needs anything
outside the store: how many *different* questions a document has been retrieved for, which a
caller passes in as `seen` if it keeps a log. Without one the feature is zero and the others
carry the score.

In [ ]:
#| export
def noise_features(db,                 # an open Database
                   store:str='store',  # the chunk table
                   seen:dict=None,     # doc_id -> the set of queries it was retrieved for, if logged
                   k:int=10,           # neighbours per chunk
                   topics:int=64,      # topic centroids for the spread features
                   tau:float=0.9,      # similarity floor for "this is a duplicate"
                   temp:float=8.0,     # softmax temperature for chunk-to-topic spread
                   limit:int=None,     # sample this many chunks instead of using all of them
                   exact_max:int=30_000,
                   seed:int=0,
                   dtype=np.float16,   # the width the store holds
) -> AttrDict:
    """Per-document noise features, computed from the vectors already in the file."""
    ids, dids, texts, V = db.chunk_matrix(store=store, limit=limit, seed=seed, dtype=dtype)
    names = list(NOISE_FEATURES)
    if not len(V): return AttrDict(doc_ids=L(), X=np.zeros((0, len(names))), names=names)

    idx, sim = _knn(V, k=k, exact_max=exact_max)
    dida = np.array(dids, dtype=object)
    # a hub is a chunk that turns up in everyone else's neighbour list
    hub = np.bincount(idx.reshape(-1), minlength=len(V)).astype(np.float32) if idx.size else np.zeros(len(V), np.float32)
    if idx.size:
        nb_doc = dida[idx]                                   # whose document each neighbour is in
        same = nb_doc == dida[:, None]
        close = sim >= tau
        dup_out = (close & ~same).any(1).astype(np.float32)  # near-duplicate of another document
        redund = (close & same).any(1).astype(np.float32)    # the document repeating itself
    else: dup_out = redund = np.zeros(len(V), np.float32)

    # boilerplate sits far from the corpus mean; proximity is a noise signal (0.12 AUC inverted)
    cen = V.mean(0); cen /= np.linalg.norm(cen) + 1e-9
    off_centre = 1.0 - V @ cen
    C_ = _centroids(V, k=topics, seed=seed)
    S = V @ C_.T
    P = np.exp(temp * (S - S.max(1, keepdims=True))); P /= P.sum(1, keepdims=True)
    spread_chunk, assign = _entropy(P), S.argmax(1)

    idf = chunk_idf(texts)
    lo_idf = np.array([-np.mean([idf.get(w, 0.0) for w in _TOK.findall((t or '').lower())] or [0.0])
                       for t in texts], np.float32)
    short = np.array([len(t or '') < 200 for t in texts], np.float32)

    # how many *different* questions this document has been retrieved for, if anything is logged
    seen = {d: set(qs) for d, qs in (seen or {}).items()}
    nq = max(len({q for s in seen.values() for q in s}), 1)

    by = defaultdict(list)
    for i, d in enumerate(dids): by[d].append(i)
    hz = (hub - hub.mean()) / (hub.std() + 1e-9)
    doc_ids, rows = L(), []
    for d, ix in by.items():
        ix = np.array(ix)
        cl = Counter(assign[ix].tolist())
        h = np.array([cl[c] for c in sorted(cl)], np.float32); h /= h.sum()
        rows.append([float(hz[ix].mean()), float(dup_out[ix].mean()), float(off_centre[ix].mean()),
                     float(spread_chunk[ix].mean()), float(_entropy(h[None, :])[0]) if len(h) > 1 else 0.0,
                     float(lo_idf[ix].mean()), float(redund[ix].mean()), float(short[ix].mean()),
                     len(seen.get(d, ())) / nq])
        doc_ids.append(d)
    return AttrDict(doc_ids=doc_ids, X=np.array(rows, np.float32), names=names)

def _robust_z(X:np.ndarray) -> np.ndarray:
    """Rank-normalise features to ~[-1.7, 1.7]. Rank beats z here: one hub outlier dropped AUC 0.988→0.55."""
    X = np.asarray(X, np.float64)
    if len(X) < 3: return np.zeros_like(X)
    out = np.empty_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        order = np.argsort(col, kind='mergesort')
        r = np.empty(len(col)); r[order] = np.arange(len(col), dtype=float)
        u = np.unique(col)
        if len(u) > 1:                       # average the ranks of ties, so constants stay constant
            for v in u[np.array([(col == v).sum() > 1 for v in u])]:
                m = col == v; r[m] = r[m].mean()
        out[:, j] = 0.0 if len(u) == 1 else (r/(len(col)-1) - 0.5) * 3.46
    return out

def noise_scores(db,                 # an open Database
                 store:str='store',  # the chunk table
                 weights:dict=None,  # per-feature blend; None -> `NOISE_W`
                 ranker=None,        # a fitted `Ranker` instead of a fixed blend
                 prefix:str=None,    # the tree prefix, for the docs table; None -> from `store`
                 **kw                # forwarded to `noise_features`
                 ) -> L:
    """Every document, most suspicious first, with the features that put it there."""
    f = noise_features(db, store=store, **kw)
    if not len(f.doc_ids): return L()
    Z = _robust_z(f.X)
    if ranker is not None: s = ranker.score(Z)
    else:
        w = np.array([(weights or NOISE_W).get(n, 0.0) for n in f.names], np.float32)
        s = Z @ w / (np.abs(w).sum() or 1.0)
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    titles = {r['id']: r['title'] for r in db.t[f'{p}docs'](select='id, title')}
    out = L(AttrDict(doc_id=d, title=titles.get(d, d), score=float(sc),
                     **{n: float(v) for n, v in zip(f.names, x)})
            for d, sc, x in zip(f.doc_ids, s, Z))
    return out.sorted(key=lambda r: -r.score)

## The ranker

A logistic model over standardised features, fitted with plain gradient descent and an L2 penalty.
It is used two ways: as a *noise* model over the features above, and as a *pairwise* re-ranker
over retrieval features. Both are the same twelve lines of arithmetic.

In [ ]:
#| export
def _sig(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

class Ranker:
    """Pairwise linear learning-to-rank over the feedback log."""
    def __init__(self, names, w=None, mu=None, sd=None, meta=None):
        self.names, self.w = list(names), None if w is None else np.asarray(w, np.float64)
        self.mu = None if mu is None else np.asarray(mu, np.float64)
        self.sd = None if sd is None else np.asarray(sd, np.float64)
        self.meta = dict(meta or {})

    def __repr__(self):
        if self.w is None: return f'Ranker({len(self.names)} features, unfitted)'
        top = sorted(zip(self.names, self.w), key=lambda t: -abs(t[1]))[:5]
        return 'Ranker(' + ', '.join(f'{n}={v:+.2f}' for n, v in top) + ', ...)'

    def _pairs(self, X, groups, y, wt, max_pairs, seed):
        by = defaultdict(list)
        for i, g in enumerate(groups): by[g].append(i)
        pi, pj, pw = [], [], []
        for ix in by.values():
            for a in ix:
                for b in ix:
                    if y[a] > y[b]: pi.append(a); pj.append(b); pw.append(min(wt[a], wt[b]))
        if not pi: return None
        pi, pj, pw = np.array(pi), np.array(pj), np.array(pw, np.float64)
        if len(pi) > max_pairs:
            s = np.random.default_rng(seed).choice(len(pi), max_pairs, replace=False)
            pi, pj, pw = pi[s], pj[s], pw[s]
        return pi, pj, pw

    def fit(self, X, groups, y, weights=None, l2:float=1.0, iters:int=25, tol:float=1e-7,
            max_pairs:int=200_000, seed:int=0):
        "Fit by IRLS on the pairwise logistic loss. Returns self, or self unfitted if no pair exists."
        X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
        wt = np.ones(len(X)) if weights is None else np.asarray(weights, np.float64)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-9
        Z = (X - self.mu) / self.sd
        p = self._pairs(Z, groups, y, wt, max_pairs, seed)
        if p is None:
            warnings.warn('no ordered pair in the feedback: nothing to fit'); return self
        pi, pj, pw = p
        D, w = Z[pi] - Z[pj], np.zeros(Z.shape[1])
        n, I = len(D), np.eye(Z.shape[1])
        for _ in range(iters):
            mu = _sig(D @ w)
            g = D.T @ (pw * (mu - 1.0)) / n + l2 * w          # every pair's target is 1
            H = (D * (pw * mu * (1 - mu))[:, None]).T @ D / n + l2 * I
            step = np.linalg.solve(H, g)
            w -= step
            if np.linalg.norm(step) < tol: break
        self.w = w
        self.meta.update(n_pairs=int(n), n_rows=int(len(X)), n_groups=len(set(groups)), l2=l2)
        return self

    def score(self, X):
        "Higher is better. Unfitted, everything ties: so a caller can always ask."
        X = np.asarray(X, np.float64)
        if self.w is None: return np.zeros(len(X))
        return ((X - self.mu) / self.sd) @ self.w

    __call__ = score
    def weights(self) -> L:
        "The fitted weights, largest first: the thing you read before trusting any of this."
        if self.w is None: return L()
        return L(sorted(zip(self.names, self.w.tolist()), key=lambda t: -abs(t[1]))).map(
            lambda t: AttrDict(feature=t[0], weight=round(t[1], 4)))

    def to_dict(self): return dict(names=self.names, w=None if self.w is None else self.w.tolist(),
                                   mu=None if self.mu is None else self.mu.tolist(),
                                   sd=None if self.sd is None else self.sd.tolist(), meta=self.meta)
    @classmethod
    def from_dict(cls, d): return cls(d['names'], d.get('w'), d.get('mu'), d.get('sd'), d.get('meta'))

## Tests

In [ ]:
from fastcore.test import test_eq, test_close
from litesearch import Index, hash_embed

class HashEmbed:
    "`hash_embed` behind an `.encode`, so these tests need no model and no network."
    def __init__(self, dims=256): self.dims = dims
    def encode(self, xs, **kw): return hash_embed(list(xs), ndim=self.dims)

In [ ]:
#| hide
# a corpus with obvious junk in it: five real documents and three copies of a cookie banner
BANNER = ('Cookie policy. All rights reserved. Privacy. Terms. Contact us. '
          'This site uses cookies to improve your experience.')
REAL = ['Reciprocal rank fusion merges two ranked lists without sharing a vector space.',
        'A static embedder indexes about seventeen hundred times cheaper than a transformer.',
        'The document tree turns a chunk hit into a section a model can read.',
        'HNSW trades a little recall for a lot of query latency.',
        'Late chunking embeds the whole document and then splits the token vectors.']
ix = Index(encoder=HashEmbed())
ix.add({f'real-{i}': t for i, t in enumerate(REAL)})
ix.add({f'junk-{i}': BANNER for i in range(3)})
test_eq(len(ix.docs), 8)

In [ ]:
#| hide
# every feature comes back, one row per document, oriented larger-is-noisier
f = ix.db.chunk_matrix(store=ix.name)
test_eq(len(f), 4)
nf = noise_features(ix.db, store=ix.name, k=3, topics=4)
test_eq(nf.names, list(NOISE_FEATURES))
test_eq(nf.X.shape, (8, len(NOISE_FEATURES)))
test_eq(len(nf.doc_ids), 8)

In [ ]:
#| hide
# the three copies of the banner sort to the top. A hashing encoder is lexical only, so the
# separation here is weaker than the 0.988 AUC a real encoder gives: the three land in the top
# four rather than the top three.
sc = noise_scores(ix.db, store=ix.name, k=3, topics=4)
rank = {r.title: i for i, r in enumerate(sc)}
junk, real = [rank[f'junk-{i}'] for i in range(3)], [rank[f'real-{i}'] for i in range(5)]
assert max(junk) <= 3, sorted(junk)
assert sum(junk)/3 < sum(real)/5 - 1, (junk, real)
test_eq(sorted(sc[0]), sorted(['doc_id', 'score', 'title', *NOISE_FEATURES]))

In [ ]:
#| hide
# `seen` is the only outside input, and it is optional: without it `promiscuity` is flat
a = noise_features(ix.db, store=ix.name, k=3, topics=4)
test_eq(set(a.X[:, a.names.index('promiscuity')]), {0.0})
b = noise_features(ix.db, store=ix.name, k=3, topics=4,
                   seen={a.doc_ids[0]: {'q1', 'q2'}, a.doc_ids[1]: {'q1'}})
test_close(b.X[0, b.names.index('promiscuity')], 1.0)
test_close(b.X[1, b.names.index('promiscuity')], 0.5)

In [ ]:
#| hide
# the ranker: fit it on the labels the scores imply, and it puts the same three on top
lab = {r.doc_id: r.title.startswith('junk') for r in sc}
X = _robust_z(nf.X)
y = np.array([float(lab[d]) for d in nf.doc_ids])
rk = Ranker(nf.names).fit(X, ['all'] * len(X), y, l2=1.0)   # one group: noise is not per-query
order = [nf.doc_ids[i] for i in np.argsort(-rk.score(X))]
assert all(lab[d] for d in order[:3]), [lab[d] for d in order]
# it round-trips through a dict, which is how a caller stores it
test_eq(Ranker.from_dict(rk.to_dict()).score(X).tolist(), rk.score(X).tolist())

In [ ]:
#| hide
# an empty store scores nothing rather than raising
empty = Index(encoder=HashEmbed())
test_eq(noise_features(empty.db, store=empty.name).X.shape, (0, len(NOISE_FEATURES)))
test_eq(noise_scores(empty.db, store=empty.name), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()